Ten notebook jest przeznaczony do badania algorytmów na zbiorach tekstowych **BBC News**

# BBC News Embeddings

Ten zbiór danych składa się z 2225 dokumentów ze strony BBC news odnoszących się do 5 obszrów tematycznych z lat 2004-2005


1.  **Wektoryzacja tesktów** tworzymy reprezentacje wektorowe na dwa sposoby, źeby pokazać,
 źe na późniejszą redukcję wymiarów duźy wpływ ma tekźe awybór algorytmu, który tworzt wektory

- paragraph-embedddingi MiniLM https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

- TF-IDF

2.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from utils.scores import Dataset

## Załadowanie danych

In [ ]:
bbc_data = pd.read_csv('./repo/data/bbc-news-data.csv', sep='\t')
bbc_data.head()

In [ ]:

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
embeddings = []
for category, content in bbc_data[['category', 'content']].values:
    embedding = model.encode(content)
    embeddings.append(embedding)


bbc_data['embedding'] = embeddings
bbc_data.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(bbc_data['content'])

bbc_data['tdidf_embedding'] = list(tfidf_matrix.toarray())
bbc_data.head()

# LDA

In [ ]:
import numpy as np
from utils.lda import my_lda
import pandas as pd
import sys
sys.modules['utils.lda'].pd = pd
lda = my_lda(n_components=2)

X_text = np.array(bbc_data['embedding'].tolist())
y_text = bbc_data['category'].values

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_text, y_text, test_size=0.2, random_state=42)
X_train_lda, _ = lda.fit(X_train, y_train)

X_test_lda = np.dot(X_test, model_text_lda.lda_components)

In [ ]:
import matplotlib.pyplot as plt
unique_categories = sorted(list(set(y_test)))
mapper = plt.cm.get_cmap('viridis', len(unique_categories))

plt.figure(figsize=(8, 6))

for i, category in enumerate(unique_categories):
    indices = (y_test == category)

    plt.scatter(
        X_test_lda[indices, 0],
        X_test_lda[indices, 1],
        label=category,
        color=mapper(i),
        alpha=0.8,
        s=40
    )

plt.title("LDA na BBC News", fontsize=13, fontweight='bold')
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend(title="Kategorie")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.manifold import trustworthiness

trust_lda = trustworthiness(X_test, X_test_lda, n_neighbors=5)
print(f"Trustworthiness dla LDA: {trust_lda:.4f}")

In [ ]:
from sklearn.metrics import normalized_mutual_info_score, silhouette_score, calinski_harabasz_score
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=len(set(y_test)),
    random_state=1,
    n_init=10
)

clusters = kmeans.fit_predict(X_test_lda)

nmi_pca = normalized_mutual_info_score(
    y_test,
    clusters
)

sil_pca = silhouette_score(
    X_test_lda,
    clusters
)

cal_har = calinski_harabasz_score(
    X_test_lda,
    clusters
)
print(f"NMI dla LDA: {nmi_pca:.4f}")
print(f"Silhouette dla LDA: {sil_pca:.4f}")
print(f"Calinski-Harabasz dla LDA: {cal_har:.4f}")

# PCA

In [ ]:
import numpy as np
from utils.pca import my_pca
import pandas as pd
import sys
sys.modules['utils.pca'].pd = pd
pca = my_pca(n_components=2)

X_text = np.array(bbc_data['embedding'].tolist())
y_text = bbc_data['category'].values

X_pca, df_pca = pca.fit(X_text)

In [ ]:
import matplotlib.pyplot as plt

mapper = plt.cm.get_cmap('viridis', len(set(bbc_data['category'])))

plt.figure(figsize=(8, 6))
for i, category in enumerate(set(bbc_data['category'])):
    indices = bbc_data['category'] == category
    plt.scatter(X_pca[indices, 0], X_pca[indices, 1], label=category, color=mapper(i), alpha=0.7)

plt.title("PCA na BBC News", fontsize=12)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.manifold import trustworthiness

trust_lda = trustworthiness(X_text, X_pca, n_neighbors=5)
print(f"Trustworthiness dla PCA: {trust_lda:.4f}")

In [ ]:
from sklearn.metrics import normalized_mutual_info_score, silhouette_score, calinski_harabasz_score
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=10,
    random_state=1,
    n_init=10
)

clusters = kmeans.fit_predict(X_pca)

nmi_pca = normalized_mutual_info_score(
    y_text,
    clusters
)

sil_pca = silhouette_score(
    X_pca,
    clusters
)

cal_har = calinski_harabasz_score(
    X_pca,
    clusters
)
print(f"NMI dla PCA: {nmi_pca:.4f}")
print(f"Silhouette dla PCA: {sil_pca:.4f}")
print(f"Calinski-Harabasz dla PCA: {cal_har:.4f}")

## Kernel PCA

In [ ]:
import matplotlib.pyplot as plt
mapper = plt.cm.get_cmap('viridis', len(set(bbc_data['category'])))
for i, category in enumerate(set(bbc_data['category'])):
    indices = bbc_data['category'] == category
    plt.scatter(X_pca_2d[indices, 0], X_pca_2d[indices, 1], label=category, color=mapper(i))
plt.legend()